# Lab 03 Homework

## 1. Data reading

Let's first see if every file is in the right place

In [2]:
from pathlib import Path

import polars as pl

DATA_DIR = Path("data")
monthly_files = [DATA_DIR / f"{month}.parquet" for month in range(1, 13)]
missing_files = [path for path in monthly_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Missing monthly Parquet files: "
        + ", ".join(str(path) for path in missing_files)
    )

zone_file = DATA_DIR / "taxi_zone_lookup.csv"
if not zone_file.is_file():
    raise FileNotFoundError(f"Missing taxi zone lookup file: {zone_file}")

Load the 2024 monthly taxi trip files and the taxi zone lookup as lazy Polars queries. Keep rides that start during 2024 and end no later than January 1, 2025

In [ ]:


rides_lf = pl.concat(
    [
        pl.scan_parquet(path).with_columns(
            pl.col("tpep_pickup_datetime").cast(pl.Datetime("ns")),
            pl.col("tpep_dropoff_datetime").cast(pl.Datetime("ns")),
        )
        for path in monthly_files
    ],
    how="vertical_relaxed",
)

pickup = pl.col("tpep_pickup_datetime")
dropoff = pl.col("tpep_dropoff_datetime")
rides_lf = (
    rides_lf
    .filter(
        (pickup >= pl.datetime(2024, 1, 1))
        & (pickup < pl.datetime(2025, 1, 1))
        & (dropoff <= pl.datetime(2025, 1, 1))
    )
)

zones_lf = (
    pl.scan_csv(zone_file)
)

Both objects are lazy. Let's their schemas and collect only a small preview

In [3]:
from IPython.display import display

print(rides_lf.collect_schema())
print(zones_lf.collect_schema())
display(rides_lf.head().collect())
display(zones_lf.head().collect())

Schema({'VendorID': Int16, 'tpep_pickup_datetime': Datetime(time_unit='ns', time_zone=None), 'tpep_dropoff_datetime': Datetime(time_unit='ns', time_zone=None), 'passenger_count': Int16, 'trip_distance': Float64, 'RatecodeID': Int16, 'store_and_fwd_flag': Categorical(ordering='physical'), 'PULocationID': Int16, 'DOLocationID': Int16, 'payment_type': Int16, 'fare_amount': Float64, 'extra': Float64, 'mta_tax': Float64, 'tip_amount': Float64, 'tolls_amount': Float64, 'improvement_surcharge': Float64, 'total_amount': Float64, 'congestion_surcharge': Float64, 'Airport_fee': Float64})
Schema({'LocationID': Int16, 'Borough': Categorical(ordering='physical'), 'Zone': Categorical(ordering='physical'), 'service_zone': Categorical(ordering='physical')})


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i16,datetime[ns],datetime[ns],i16,f64,i16,cat,i16,i16,i16,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


LocationID,Borough,Zone,service_zone
i16,cat,cat,cat
1,"""EWR""","""Newark Airport""","""EWR"""
2,"""Queens""","""Jamaica Bay""","""Boro Zone"""
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone"""
4,"""Manhattan""","""Alphabet City""","""Yellow Zone"""
5,"""Staten Island""","""Arden Heights""","""Boro Zone"""
